# L13 demo: two surrogates, and whether their uncertainty means anything

We build two surrogates for the **NASA airfoil self-noise** dataset, a Gaussian
process and a deep ensemble of PyTorch mean-variance networks, and then spend
most of the notebook on the harder question: are the uncertainties they report
any good?

The three things to watch:

1. The dataset has **five feature columns and four independent knobs**. The
   displacement thickness is determined by the other three. If you sample a
   design space that does not exist, your surrogate will confidently answer
   questions about it.
2. The GP's posterior standard deviation **grows** when you ask it about a
   free-stream velocity it never saw. The ensemble's grows much less.
3. Split conformal prediction has a coverage guarantee. We break it three
   different ways, in increasing order of realism.

**Requirements.** `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`,
`torch`, and `mlflow`.

## The dataset, and the design space behind it

1,503 one-third-octave measurements from an anechoic wind tunnel, from Brooks,
Pope and Marcolini's 1989 NASA report. Five inputs, one output: scaled sound
pressure level in decibels.

This is a surrogate problem in the strict sense. Each row is not a free
observation; it is a slice of a tunnel run that cost setup, instrumentation and
staff time. If a model can predict the spectrum for a configuration nobody has
run, that is a run you do not have to pay for.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)
LOCAL = CACHE / 'airfoil_self_noise.dat'
URL = 'https://archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip'

if not LOCAL.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    LOCAL.write_bytes(archive.read('airfoil_self_noise.dat'))

print('cached:', LOCAL, f'({LOCAL.stat().st_size / 1e3:.0f} kB)')

In [ ]:
import numpy as np
import pandas as pd

COLS = ['freq_hz', 'aoa_deg', 'chord_m', 'velocity_ms', 'thickness_m', 'spl_db']
RAW = COLS[:5]

air = pd.read_csv(LOCAL, sep='\t', header=None, names=COLS)
air['strouhal'] = air.freq_hz * air.thickness_m / air.velocity_ms

print(air.shape)
print(air.describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())

### Count the knobs before you count the columns

A tunnel configuration is an (angle of attack, chord, free-stream velocity)
setting. Within a configuration the frequency sweeps. The suction-side
displacement thickness is *not* something the operator sets: it is the boundary
layer that results, predicted from the other three.

Check that, because the whole design-of-experiments discussion depends on it.

In [ ]:
groups = air.groupby(['aoa_deg', 'chord_m', 'velocity_ms']).ngroup().to_numpy()
per_config = air.groupby(['aoa_deg', 'chord_m', 'velocity_ms']).thickness_m.nunique()

print(f'{groups.max() + 1} distinct configurations')
print(f'thickness takes one value in {(per_config == 1).sum()}/{len(per_config)} '
      f'of them: five columns, four knobs')

levels = {c: air[c].nunique() for c in ['freq_hz', 'aoa_deg', 'chord_m', 'velocity_ms']}
full = np.prod(list(levels.values()))
print(levels)
print(f'a full factorial would be {full:,} runs; we have {len(air):,} '
      f'({len(air) / full:.1%} of the grid)')

## Three splits, in increasing order of honesty

- **random rows** puts most of a frequency sweep in training and asks the model
  to fill in the rest. That is interpolation along a smooth curve, and it is the
  number that flatters you.
- **held-out configurations** holds out whole tunnel runs, so the model is asked
  for a sweep it has never seen. This is the L9 lesson applied here.
- **held-out velocity** removes every row at 71.3 m/s. That is not a harder
  interpolation problem; it is extrapolation, and it is exactly what a design
  loop does to a surrogate.

In [ ]:
from sklearn.model_selection import KFold

X_raw = air[RAW].to_numpy(float)
y = air.spl_db.to_numpy()


def grouped_split(groups, n_splits=5, fold=0):
    '''Hold out every n_splits-th configuration, by sorted group id.

    Deliberately not `GroupKFold`. See the note below.
    '''
    unique = np.unique(groups)
    held_out = unique[fold::n_splits]
    mask = np.isin(groups, held_out)
    return np.where(~mask)[0], np.where(mask)[0]


splits = {}
splits['random rows'] = next(iter(KFold(5, shuffle=True, random_state=0).split(X_raw)))
splits['held-out configurations'] = grouped_split(groups)
fast = air.velocity_ms.to_numpy() == 71.3
splits['held-out velocity'] = (np.where(~fast)[0], np.where(fast)[0])

for name, (tr, te) in splits.items():
    print(f'{name:26s} {len(tr):5d} train  {len(te):4d} test  '
          f'(predict the mean: RMSE '
          f'{np.sqrt(np.mean((y[te] - y[tr].mean()) ** 2)):.2f} dB)')

:::{admonition} Why not `GroupKFold`?
:class: warning

Because it is not reproducible across scikit-learn versions. 1.8 and 1.9 assign
groups to folds by different rules, with the same signature and the same
`shuffle=False`, and nothing warns you.

On this dataset that single change moved the GP's held-out RMSE from **2.08 dB
to 1.50 dB** and its 95% coverage from **91% to 94%** — larger than most of the
effects this whole session measures. Every number in the notes comes from the
pinned split above, so run it that way if you want to reproduce them.

This is L1's `np.trapz` lesson in a new costume, and the fix is the same one:
pin the thing you depend on rather than inherit it.
:::

## Surrogate one: a Gaussian process

A GP is the default surrogate for low-dimensional, smooth, expensive problems,
and the reason is not accuracy: it is that the posterior variance comes out of
the same algebra as the mean, for free, without a second model or a second
training run.

The kernel choice is the modelling choice. `Matern(nu=2.5)` assumes the response
is twice differentiable, which is a weaker and usually safer assumption than the
RBF kernel's infinite smoothness. A separate length scale per input (ARD) lets
the fit tell you which variables matter, which is worth reading afterwards.

`WhiteKernel` is the aleatoric part: it is the GP learning how much of the
scatter is measurement noise that no amount of data will remove.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler


def fit_gp(X, y):
    kernel = (ConstantKernel(1.0, (1e-3, 1e4))
              * Matern(length_scale=np.ones(X.shape[1]), nu=2.5,
                       length_scale_bounds=(1e-2, 1e3))
              + WhiteKernel(1e-2, (1e-6, 1e2)))
    return GaussianProcessRegressor(kernel=kernel, normalize_y=True, alpha=0.0,
                                    random_state=0).fit(X, y)


tr, te = splits['held-out configurations']
scaler = StandardScaler().fit(X_raw[tr])
gp = fit_gp(scaler.transform(X_raw[tr]), y[tr])

mu_gp, sd_gp = gp.predict(scaler.transform(X_raw[te]), return_std=True)
print(gp.kernel_)
print(f'\nGP RMSE {np.sqrt(np.mean((y[te] - mu_gp) ** 2)):.2f} dB, '
      f'mean posterior sigma {sd_gp.mean():.2f} dB')

Read the learned length scales. A short length scale means the response changes
quickly along that input, so the model needs points close together there; a
length scale pinned at its upper bound means the GP has decided that input does
not matter. That is a sampling plan for the next campaign, delivered for free.

In [ ]:
lengths = gp.kernel_.k1.k2.length_scale
for name, ell in sorted(zip(RAW, lengths), key=lambda p: p[1]):
    print(f'  {name:14s} length scale {ell:8.3f}  '
          f'{"varies fast" if ell < 1 else "nearly ignored" if ell > 20 else ""}')
print(f'\nlearned noise level: {gp.kernel_.k2.noise_level:.3f} '
      f'(sd {np.sqrt(gp.kernel_.k2.noise_level):.2f} dB) -- the aleatoric part')

## Surrogate two: a deep ensemble

Lakshminarayanan, Pritzel and Blundell's recipe, in about thirty lines. Each
member predicts a **mean and a variance** and is trained by Gaussian negative
log-likelihood rather than mean squared error, so it can say "I am unsure here"
about individual points. Train five of them from different random
initialisations.

The two uncertainties then fall out of the law of total variance:

$$\underbrace{\mathrm{Var}[y]}_{\text{total}}
= \underbrace{\mathbb{E}_m[\sigma_m^2]}_{\text{aleatoric}}
+ \underbrace{\mathrm{Var}_m[\mu_m]}_{\text{epistemic}}$$

The members' average predicted variance is the noise they all agree is there.
The spread *between* their means is their disagreement, which is what more data
can fix.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(4)


class MeanVariance(nn.Module):
    def __init__(self, d_in, width=64, p_drop=0.0):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, width), nn.ReLU(), nn.Dropout(p_drop))
        self.head = nn.Linear(width, 2)

    def forward(self, x):
        out = self.head(self.body(x))
        # softplus keeps the variance positive: a hard constraint, for free.
        return out[:, :1], nn.functional.softplus(out[:, 1:]) + 1e-3


def fit_member(X, z, seed, epochs=400, p_drop=0.0):
    torch.manual_seed(seed)
    model = MeanVariance(X.shape[1], p_drop=p_drop)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    xt = torch.tensor(X, dtype=torch.float32)
    zt = torch.tensor(z, dtype=torch.float32)[:, None]
    gen = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        for idx in torch.randperm(len(xt), generator=gen).split(64):
            mu, var = model(xt[idx])
            loss = (torch.log(var) / 2 + (zt[idx] - mu) ** 2 / (2 * var)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model.eval()


def fit_ensemble(X, y, n_members=5, seed0=0, **kw):
    y_mean, y_std = y.mean(), y.std()
    members = [fit_member(X, (y - y_mean) / y_std, seed0 + i, **kw)
               for i in range(n_members)]
    return members, y_mean, y_std


def ensemble_predict(members, X, y_mean, y_std):
    xt = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        out = [(m(xt)[0].numpy().ravel(), m(xt)[1].numpy().ravel()) for m in members]
    mus = np.array([o[0] for o in out])
    variances = np.array([o[1] for o in out])
    epistemic, aleatoric = mus.var(0), variances.mean(0)
    return (mus.mean(0) * y_std + y_mean,
            np.sqrt(epistemic + aleatoric) * y_std,
            np.sqrt(epistemic) * y_std,
            np.sqrt(aleatoric) * y_std)


members, ym, ys = fit_ensemble(scaler.transform(X_raw[tr]), y[tr])
mu_en, sd_en, epi_en, ale_en = ensemble_predict(
    members, scaler.transform(X_raw[te]), ym, ys)

print(f'ensemble RMSE {np.sqrt(np.mean((y[te] - mu_en) ** 2)):.2f} dB, '
      f'mean sigma {sd_en.mean():.2f} dB')
print(f'  epistemic {epi_en.mean():.2f} dB   aleatoric {ale_en.mean():.2f} dB')

## What the two look like on a configuration neither model saw

Pick the longest held-out frequency sweep and plot both surrogates with 95%
bands. The point predictions are close. The bands are not.

In [ ]:
import matplotlib.pyplot as plt

target = pd.Series(groups[te]).value_counts().idxmax()
mask = groups[te] == target
order = np.argsort(air.freq_hz.to_numpy()[te][mask])
freq = air.freq_hz.to_numpy()[te][mask][order]
truth = y[te][mask][order]
cfg = air.iloc[te[mask][0]]

fig, ax = plt.subplots(figsize=(9, 5.2))
for mu, sd, colour, label in ((mu_gp, sd_gp, '#1f5c99', 'Gaussian process'),
                              (mu_en, sd_en, '#c41230', 'deep ensemble')):
    m, s = mu[mask][order], sd[mask][order]
    ax.plot(freq, m, color=colour, lw=2.2, label=label)
    ax.fill_between(freq, m - 1.96 * s, m + 1.96 * s, color=colour, alpha=0.16, lw=0)
ax.plot(freq, truth, 'o', color='#1a1a1a', ms=6, label='wind tunnel', zorder=5)
ax.set_xscale('log')
ax.set_xlabel('Frequency, Hz')
ax.set_ylabel('Sound pressure level, dB')
ax.set_title(f'{cfg.aoa_deg:g} deg, chord {cfg.chord_m:g} m, {cfg.velocity_ms:g} m/s')
ax.legend(frameon=False)
ax.grid(True, which='both', lw=0.6, color='#d8d8d8')
plt.show()

## Does the model notice when it leaves the data?

This is the question a design loop turns on. Refit both surrogates with every
71.3 m/s row removed, predict those rows, and compare the *reported* sigma to
the sigma on the honest interpolation split.

A surrogate that gets worse and says so is usable. A surrogate that gets worse
quietly is a trap, because an optimiser will walk straight into the region where
it is wrong and confident.

In [ ]:
def evaluate(split_name):
    tr, te = splits[split_name]
    sc = StandardScaler().fit(X_raw[tr])
    Xtr, Xte = sc.transform(X_raw[tr]), sc.transform(X_raw[te])

    g = fit_gp(Xtr, y[tr])
    gmu, gsd = g.predict(Xte, return_std=True)

    mem, m0, s0 = fit_ensemble(Xtr, y[tr])
    emu, esd, epi, ale = ensemble_predict(mem, Xte, m0, s0)

    def row(mu, sd):
        return dict(rmse=np.sqrt(np.mean((y[te] - mu) ** 2)),
                    sigma=sd.mean(),
                    picp=np.mean(np.abs(y[te] - mu) <= 1.96 * sd),
                    width=np.mean(2 * 1.96 * sd))

    return {'GP': row(gmu, gsd), 'ensemble': row(emu, esd),
            'epistemic': epi.mean(), 'aleatoric': ale.mean()}


report = {name: evaluate(name) for name in splits}

print(f'{"split":26s} {"model":10s} {"RMSE":>6s} {"sigma":>7s} '
      f'{"PICP95":>8s} {"width":>7s}')
for name, res in report.items():
    for model in ('GP', 'ensemble'):
        r = res[model]
        print(f'{name:26s} {model:10s} {r["rmse"]:6.2f} {r["sigma"]:7.2f} '
              f'{r["picp"]:7.1%} {r["width"]:7.2f}')

In [ ]:
growth_gp = report['held-out velocity']['GP']['sigma'] / \
    report['held-out configurations']['GP']['sigma']
growth_en = report['held-out velocity']['ensemble']['sigma'] / \
    report['held-out configurations']['ensemble']['sigma']
print(f'leaving the training envelope multiplies the reported sigma by')
print(f'  GP:       {growth_gp:.2f}x')
print(f'  ensemble: {growth_en:.2f}x')
print()
print('epistemic vs aleatoric, by split:')
for name, res in report.items():
    print(f'  {name:26s} epistemic {res["epistemic"]:.2f} dB   '
          f'aleatoric {res["aleatoric"]:.2f} dB')

The GP's variance grows because a stationary kernel has nowhere else to go: far
from any training point the posterior relaxes back to the prior, so the
uncertainty is large by construction. That property is what makes GPs the
default surrogate inside a Bayesian optimisation loop, which is L14.

It is also why a GP is *conservative* rather than *correct* out of distribution.
Wide intervals are not the same as right intervals, which is the next section.

## Checking the uncertainty: coverage and sharpness

Two numbers, and you need both.

**PICP**, the prediction-interval coverage probability, is the fraction of test
points inside the nominal 95% interval. If it is 76%, the model is overconfident
and every risk calculation downstream is wrong.

**Width** is the average size of the interval. An interval of plus or minus 40 dB
covers everything and tells you nothing. Gneiting, Balabdaoui and Raftery's
formulation is the one to remember: maximise **sharpness subject to
calibration**. Get the coverage right first, then make the interval as narrow as
you can.

A reliability diagram checks coverage at every level at once, not just 95%.

In [ ]:
from scipy.stats import norm

nominal = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99])
tr, te = splits['held-out configurations']

fig, ax = plt.subplots(figsize=(6.2, 5.6))
for mu, sd, colour, label in ((mu_gp, sd_gp, '#1f5c99', 'Gaussian process'),
                              (mu_en, sd_en, '#c41230', 'deep ensemble')):
    empirical = [np.mean(np.abs(y[te] - mu) <= norm.ppf(0.5 + p / 2) * sd)
                 for p in nominal]
    ax.plot(nominal, empirical, 'o-', color=colour, lw=2, label=label)
    print(f'{label:20s} ' + '  '.join(
        f'{p:.0%}->{e:.0%}' for p, e in zip(nominal[-3:], empirical[-3:])))
ax.plot([0, 1], [0, 1], 'k--', lw=1.3)
ax.set_xlabel('Nominal coverage')
ax.set_ylabel('Empirical coverage')
ax.set_title('Reliability, held-out configurations')
ax.legend(frameon=False)
ax.grid(True, lw=0.6, color='#d8d8d8')
plt.show()

## Split conformal prediction, and the assumption it rests on

Conformal prediction is the one method here that comes with a finite-sample
guarantee, and the guarantee is remarkably cheap. Hold out a calibration set,
compute the absolute residuals of *any* model on it, take the appropriate
empirical quantile, and use it as a fixed-width interval. There is no
distributional assumption about the errors and no assumption about the model.

There is one assumption, and it is about the data: the calibration points and
the test points must be **exchangeable**. Read that as "drawn from the same
distribution, in no meaningful order."

Watch what happens as we make the split more realistic and that assumption gets
less true.

In [ ]:
def conformal_quantile(residuals, alpha=0.05):
    n = len(residuals)
    level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(residuals, level, method='higher'))


print(f'{"split":26s} {"PICP95":>8s} {"width":>9s}')
conformal_report = {}
for name, (tr_i, te_i) in splits.items():
    # A fresh generator per split, so each one gets the same calibration draw
    # rather than a continuation of the previous split's stream.
    perm = np.random.default_rng(0).permutation(len(tr_i))
    n_cal = len(tr_i) // 4
    cal, sub = tr_i[perm[:n_cal]], tr_i[perm[n_cal:]]

    sc = StandardScaler().fit(X_raw[sub])
    model = fit_gp(sc.transform(X_raw[sub]), y[sub])
    q = conformal_quantile(np.abs(y[cal] - model.predict(sc.transform(X_raw[cal]))))
    mu = model.predict(sc.transform(X_raw[te_i]))
    picp = np.mean(np.abs(y[te_i] - mu) <= q)
    conformal_report[name] = (picp, 2 * q)
    print(f'{name:26s} {picp:7.1%} {2 * q:8.2f} dB')

The guarantee did not fail. The assumption did, and it failed in a way nothing
in the code could detect: the calibration set was carved out of the training
rows at random, so it looks like the training data, and the test rows do not.

That is the general shape of every uncertainty failure in this session. The
method is fine. The claim it makes is conditional on something about your data,
and the something is usually the same thing: that the rows you calibrated on and
the rows you will predict came from the same place.

## Physics as a change of coordinates

Trailing-edge noise scales on the Strouhal number $St = f\,\delta^*/U$. That is
not extra information; it is a statement that frequency, boundary-layer
thickness and velocity are not three independent axes.

Refit the GP in those coordinates. Nothing is added and nothing is removed: the
same five numbers per row, re-expressed. Watch the extrapolation error.

In [ ]:
def physics_features(d):
    return np.column_stack([
        np.log10(d.freq_hz * d.thickness_m / d.velocity_ms),   # log Strouhal
        d.aoa_deg,
        np.log10(d.chord_m),
        np.log10(d.thickness_m),
        d.velocity_ms / 340.3,                                 # Mach
    ])


X_phys = physics_features(air)

print(f'{"split":26s} {"raw":>8s} {"physics":>9s}')
for name, (tr_i, te_i) in splits.items():
    row = []
    for X in (X_raw, X_phys):
        sc = StandardScaler().fit(X[tr_i])
        m = fit_gp(sc.transform(X[tr_i]), y[tr_i])
        row.append(np.sqrt(np.mean((y[te_i] - m.predict(sc.transform(X[te_i]))) ** 2)))
    print(f'{name:26s} {row[0]:7.2f} {row[1]:8.2f} dB   '
          f'{"physics wins" if row[1] < row[0] else "physics loses"}')

Try it yourself on a slice where the physics does *not* apply: hold out
`aoa_deg >= 15.4` instead of a velocity. Those are the near-stall runs, where
the noise is dominated by separation rather than by the trailing-edge mechanism
the Strouhal number describes. The notes report what happens; predict it before
you run it.

In [ ]:
stall = air.aoa_deg.to_numpy() >= 15.4
tr_i, te_i = np.where(~stall)[0], np.where(stall)[0]
print(f'{te_i.size} near-stall rows held out')
for label, X in (('raw', X_raw), ('physics', X_phys)):
    sc = StandardScaler().fit(X[tr_i])
    m = fit_gp(sc.transform(X[tr_i]), y[tr_i])
    mu, sd = m.predict(sc.transform(X[te_i]), return_std=True)
    print(f'  {label:8s} RMSE {np.sqrt(np.mean((y[te_i] - mu) ** 2)):5.2f} dB   '
          f'PICP95 {np.mean(np.abs(y[te_i] - mu) <= 1.96 * sd):5.1%}')

## Log it, or it did not happen

Everything above is a run: a split, a model family, a feature set, a seed. The
miniproject asks for a surrogate with uncertainty **tracked in MLflow**, and the
thing worth logging is not only the RMSE.

Log the coverage. A tracked experiment where every run reports RMSE and none
reports PICP will let you select a model that is accurate and overconfident,
which is the worst combination for a design loop.

In [ ]:
import mlflow

mlflow.set_experiment('l13-surrogates')

for split_name, res in report.items():
    for model_name in ('GP', 'ensemble'):
        with mlflow.start_run(run_name=f'{model_name}-{split_name}'):
            mlflow.log_params({'model': model_name, 'split': split_name,
                               'features': 'raw', 'n_members':
                               5 if model_name == 'ensemble' else 1})
            mlflow.log_metrics({
                'rmse_db': float(res[model_name]['rmse']),
                'mean_sigma_db': float(res[model_name]['sigma']),
                'picp_95': float(res[model_name]['picp']),
                'interval_width_db': float(res[model_name]['width']),
            })

print('logged. run `mlflow ui` and sort by picp_95, not by rmse_db.')

## What to take away

The GP and the ensemble agree to within a few tenths of a decibel on the point
prediction and disagree completely about how much to trust it. Under a held-out
configuration split the GP covers about 91% of a nominal 95% interval and the
five-member ensemble about 76%, and if you had only compared RMSE you would
never have seen the difference.

Then all of it degrades when you leave the training envelope, which is the one
thing a design loop guarantees you will do. The GP degrades most gracefully
because a stationary kernel reverts to its prior, and that same property is what
L14 exploits: an acquisition function is a rule for walking towards the places
where the posterior variance is large.

Before the miniproject, run this notebook once with your own dataset in place of
the airfoil one, and report PICP alongside RMSE from the first run.